# ADS-B Collector — Step by Step

This notebook explains the collector and runs it.

**Architecture:**
```
adsb.lol API  →  fetch_berlin()  →  build_document()  →  MongoDB Atlas (airline_landing.adsb_raw)
```

**Prerequisite:** `.env` at the **project root** (`airline-data-platform/.env`) with:
```
MONGO_URI_RW=mongodb+srv://airline-collector-rw:<password>@mongo-mk1.ptb1k2b.mongodb.net/airline_landing
MONGO_DB=airline_landing
```
This collector writes data, so it uses `MONGO_URI_RW` (the `airline-collector-rw` user with write access) via `from_env(write=True)`.
Credentials: `.env` is gitignored, never commit.

## Step 1 — Imports and path setup

The collector lives in `01-bronze/collectors/adsb_collector.py`.  
The MongoDB connector lives in `data_connectors/mongo.py`.  
To let Python find both, the **repo root** (for `data_connectors`) and **`01-bronze/`** (for `collectors`) must be on the search path.

In [ ]:
import sys
from pathlib import Path

# repo root (for `data_connectors`) and 01-bronze/ (for `collectors`) on the search path
root = Path.cwd()
if not (root / "data_connectors").is_dir():
    root = root.parent
for p in (root, root / "01-bronze"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

print(f"Repo root: {root}")

## Step 2 — Query the API

`fetch_berlin()` makes an HTTP GET to `adsb.lol` and returns the raw API response.  
No login required — the API is public.

In [ ]:
from collectors.adsb_collector import fetch_berlin

raw = fetch_berlin()

print(f"API timestamp (ms): {raw['now']}")
print(f"Aircraft found:     {raw['total']}")
print(f"Processing time:    {raw.get('ptime')} ms")
print(f"\nFirst 2 entries in 'ac':")
for ac in raw['ac'][:2]:
    print(ac)

## Step 3 — Inspect raw data as a DataFrame

The `ac` array contains one dict per aircraft. Key fields:

| Field | Description |
|---|---|
| `hex` | ICAO address (unique transponder identifier) |
| `flight` | Callsign (e.g. `DLH123`) |
| `lat` / `lon` | Position |
| `alt_baro` | Barometric altitude in feet, or `"ground"` |
| `gs` | Ground speed in knots |
| `r` | Registration (e.g. `D-AIZW`) |
| `t` | Aircraft type (e.g. `A320`) |

In [ ]:
import pandas as pd

df = pd.DataFrame(raw['ac'])
print(f"Shape: {df.shape}  ({df.shape[0]} aircraft, {df.shape[1]} fields)")
print(f"All fields: {list(df.columns)}\n")

show = [c for c in ['hex', 'flight', 'r', 't', 'lat', 'lon', 'alt_baro', 'gs'] if c in df.columns]
df[show].head(10)

## Step 4 — Build the document

`build_document()` wraps the raw response in the MongoDB format.  
The `ac` array is kept **unchanged** — no transformation, no cleaning.  
This is intentional: MongoDB is the raw landing zone.

In [ ]:
from collectors.adsb_collector import build_document

doc = build_document(raw)

# show everything except the ac array
meta = {k: v for k, v in doc.items() if k != 'ac'}
for k, v in meta.items():
    print(f"  {k}: {v}")
print(f"  ac: [{len(doc['ac'])} aircraft dicts]")

## Step 5 — Connect to MongoDB

`from_env(write=True)` reads `MONGO_URI_RW` and `MONGO_DB` from the `.env` file — the collector needs write access.  
The context manager (`with`) ensures the connection is always closed at the end.

In [ ]:
from data_connectors.mongo import from_env

db = from_env(write=True).connect()
print("Connected.")

## Step 6 — Write snapshot to MongoDB

`insert_adsb_snapshot()` calls `collection.insert_one(doc)` internally.  
MongoDB automatically assigns a `_id` (ObjectId) and returns it.

In [ ]:
inserted_id = db.insert_adsb_snapshot("adsb_raw", doc)
print(f"Stored with _id: {inserted_id}")

total = db.collection("adsb_raw").count_documents({})
print(f"Total snapshots in MongoDB: {total}")

## Step 7 — Collect multiple snapshots (mini loop)

For testing: collect 3 snapshots 30 seconds apart.  
In production the collector runs as a script: `python collectors/adsb_collector.py --interval 60`

In [ ]:
import time

N = 3          # number of snapshots
INTERVAL = 30  # seconds between requests

for i in range(N):
    raw_i = fetch_berlin()
    doc_i = build_document(raw_i)
    _id = db.insert_adsb_snapshot("adsb_raw", doc_i)
    print(f"[{i+1}/{N}] {doc_i['collected_at']}  —  {doc_i['total']} aircraft  →  _id={_id}")
    if i < N - 1:
        time.sleep(INTERVAL)

print("\nDone.")

## Step 8 — Close connection

In [ ]:
db.close()
print("Connection closed.")